In [0]:
import yaml

dict_rfd = {'rfdiffusion' : {
    'contigs' : "A:40-50/B5-13/20-30/C5-13/3-8",
    'pdb' : '/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular/inputs/collagen_triple_helix_alpha2_model_0.pdb',
    'iterations' : 50,
    'hotspot' : 'A12,A13,A14,A15,A74,A75,A76,A77,A78,A79,A80,A115,A116,A117',
    'num_designs' : 8,
    'visual' : 'image',
    'symmetry': 'none',
    'order' : 1,
    'chains' : '',
    'add_potential' : True,
    'partial_T' : 'auto',
    'use_beta_model' : False,
}}

dict_mpnn = {'mpnn' :{
    'num_seqs' : 8,
    'mpnn_sampling_temp' : 0.1,
    'rm_aa' : 'C',
    'use_solubleMPNN' : True,
    }}

dict_af2 = {'alphafold2' : {
    'initial_guess' : True, 
    'num_recycles' : 3,
    'use_multimer' : False
}}

dict_pipeline = dict_rfd | dict_mpnn | dict_af2

overarching_save_destination = '/'.join(dict_rfd['rfdiffusion']['pdb'].split('/')[:-2])
overarching_save_destination
path_folder_yaml = f"{overarching_save_destination}/rfdiffusion/yaml/" 
with open(path_folder_yaml + "base.yaml", 'w') as file:
    documents = yaml.dump(dict_pipeline, file)

In [0]:
# YAML File preserves the data type of the values in the dictionary
with open (path_folder_yaml + "base.yaml") as file:
    rfdiffusion_config = yaml.full_load(file)
for tool in rfdiffusion_config:
    print(tool)
    print("--------" * 10)
    for parameter, value in rfdiffusion_config[tool].items():
        print(parameter, value)
        print(type(value))
        print("--------" * 10)

### Create Combinations of Potential Linker Lengths via Cartesian Product: Alpha

In [0]:
import itertools
linker_lengths = ['20-30', '30-40', '40-50', '50-60', '60-70']

# Cartesian product of list with itself
cartesian_pairs = list(itertools.product(linker_lengths, linker_lengths))

print(f"Cartesian Product with itself yielded {len(cartesian_pairs)} pairs")
print(cartesian_pairs) 


In [0]:
for linker_length1, linker_length2 in cartesian_pairs:
    custom_rfd_contig = f"A:{linker_length1}/B5-13/{linker_length2}/C5-13/3-8"
    print("Custom RFD Contig: ", custom_rfd_contig)
    rfdiffusion_config['rfdiffusion']['contigs'] = custom_rfd_contig

    if rfdiffusion_config['rfdiffusion']['use_beta_model']:
        model_type = 'beta'
    else:
        model_type = 'alpha'
    
    print("Model Type: ", model_type)

    with open(path_folder_yaml + f"dble_glob_{model_type}_linklen1_{linker_length1}_linklen2_{linker_length2}.yaml", 'w') as file:
        documents = yaml.dump(rfdiffusion_config, file)

### Create Combinations of Potential Linker Lengths via Cartesian Product: Beta

In [0]:
linker_lengths = ['20-30', '30-40', '40-50', '50-60']

# Cartesian product of list with itself
cartesian_pairs = list(itertools.product(linker_lengths, linker_lengths))

print(f"Cartesian Product with itself yielded {len(cartesian_pairs)} pairs")
print(cartesian_pairs)

In [0]:
for linker_length1, linker_length2 in cartesian_pairs:
    custom_rfd_contig = f"A:{linker_length1}/B5-13/{linker_length2}/C5-13/3-8"
    print("Custom RFD Contig: ", custom_rfd_contig)
    rfdiffusion_config['rfdiffusion']['contigs'] = custom_rfd_contig

    # Sub in beta_model:
    rfdiffusion_config['rfdiffusion']['use_beta_model'] = True
    if rfdiffusion_config['rfdiffusion']['use_beta_model']:
        model_type = 'beta'
    else:
        model_type = 'alpha'
    
    print("Model Type: ", model_type)

    with open(path_folder_yaml + f"dble_glob_{model_type}_linklen1_{linker_length1}_linklen2_{linker_length2}.yaml", 'w') as file:
        documents = yaml.dump(rfdiffusion_config, file)

In [0]:
import os
list_yaml_names = os.listdir(path_folder_yaml)
final_yaml_names = [name for name in list_yaml_names if 'dble' in name] * 2
print(len(final_yaml_names))
final_yaml_names

In [0]:
total_designs = 82*8*8
total_designs

In [0]:
dbutils.jobs.taskValues.set(key = "final_yaml_names", value = final_yaml_names)